# Tugas Besar 2 IF3170 Inteligensi Artifisial

*Klasifikasi pembatalan pemesanan hotel (`is_canceled`).*

## IF3170_G08_Teazzi

- 13524034 - Jason Edward Salim
- 13524064 - Stefani Angeline Oroh
- 13524067 - Bryan Pratama Putra Hendra
- 13524079 - Angelina Andra Alanna
- 13524106 - Michael James Liman

## Table of Contents (CRISP-DM)

1. [**Business Understanding**](#1)
2. [**Data Understanding**](#2)
3. [**Data Preparation**](#3)
4. [**Modeling**](#4)
5. [**Evaluation**](#5)
6. [**Deployment**](#6)

## Initialization

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.insert(0, str(Path.cwd() if Path.cwd().name == "src" else Path.cwd() / "src"))
from config import Settings, seed_everything
from eda import types, missing, duplicates, pie, bar, histogram, boxplot, iqr_outliers, target_rate, correlation
from preprocessing import load_data, split, Preprocessor, Pipeline
from validation import fit, holdout, kfold, compare, tune, best, importance, confusion, report, misclassified, submit
from models import KNN, LogisticRegression, SVM, DTL, BayesNetwork
seed_everything()

In [ ]:
full_train, test = load_data()
X_train, X_val, y_train, y_val = split(full_train)
eda_train = full_train.loc[X_train.index]
pd.Series({"train": len(full_train), "train_fit": len(X_train), "validation": len(X_val), "test": len(test)})

---

# Business Understanding <a name="1"></a>

---

Memprediksi apakah sebuah pemesanan hotel akan dibatalkan (`is_canceled = 1`) atau tidak (`is_canceled = 0`). Metrik evaluasi: macro F1-score.

> TODO: konteks bisnis dan dampak false positive / false negative.

---

# Data Understanding <a name="2"></a>

---

## Data Overview

In [ ]:
eda_train

In [ ]:
display(types(eda_train))
display(types(test))

In [ ]:
missing(eda_train, test)

In [ ]:
duplicates(eda_train, test)

## Target Distribution

In [ ]:
pie(eda_train)
bar(eda_train)

## Feature Analysis

In [ ]:
num_col = eda_train.select_dtypes("number").columns.drop(Settings.TARGET)
cat_col = eda_train.columns.difference(num_col).drop([Settings.ID, Settings.TARGET])
display(eda_train[num_col].describe().T)
display(eda_train[cat_col].describe().T)

In [ ]:
histogram(eda_train, num_col)
boxplot(eda_train, num_col)

In [ ]:
iqr_outliers(eda_train, num_col)

In [ ]:
target_rate(eda_train, list(cat_col) + ['special_requests', 'arrival_month', 'is_returning_guest'])

In [ ]:
correlation(eda_train)

#### Insights

> TODO

---

# Data Preparation <a name="3"></a>

---

Split stratified memakai seed validasi terpisah dan dilakukan sebelum EDA berbasis target. EDA menggunakan bagian training saja. Di setiap fold, preprocessor di-fit hanya pada training fold; untuk submission, preprocessor di-fit ulang pada seluruh data train. Profil fitur berulang dipertahankan karena ID booking berbeda.


In [ ]:
X_test = test.drop(columns=[Settings.ID])
pd.Series({"X_train": len(X_train), "X_val": len(X_val), "X_test": len(X_test)})

In [ ]:
def make_preprocessor():
    return Preprocessor(
        zero_as_missing=[],
        num_strategy="median",
        cap=["days_before_arrival", "nightly_rate_idr"],
        ordinal={"meal_package": ["Room Only", "Breakfast", "Half Board", "Full Board"]},
        scaler="minmax",
    )
preprocessor_factory = make_preprocessor

Nilai numerik yang hilang diimputasi dengan median dan kategori dengan modus. Indikator harga nol dibuat sebelum clipping. Tipe kamar di-one-hot karena nominal; paket makan tetap ordinal. KNN memakai min-max scaling; fitur biner dan one-hot tetap 0–1. Bobot meal/room mengatur kontribusi grup tersebut pada jarak KNN. Statistik preprocessing di-fit di dalam training fold.

---

# Modeling <a name="4"></a>

---

Model from scratch ada di `src/models/` (implementasi `fit`/`predict`). Model sklearn dipakai sebagai pembanding. `METHOD` menentukan penanganan class imbalance (`None`, `'smote'`, `'undersample'`), hanya diterapkan pada data latih.

In [ ]:
METHOD = None
# Pipeline(model, select=k, pca=n) adds per-model feature selection and PCA
models = {
    "knn": KNN(k=71, metric="manhattan", vote_weights="inverse_square", feature_weights={"meal_package": 0.1, "room_category": 0.1}, threshold=0.44),
    "lr": LogisticRegression(lr=0.01, n_iter=1000),
    "svm": SVM(kernel="rbf", C=3, gamma=0.1),
    "dtl": DTL(max_depth=None),
    "bn": BayesNetwork(),
}

In [ ]:
from sklearn.linear_model import LogisticRegression as SkLogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
sk_models = {
    "sk_knn": KNeighborsClassifier(5),
    "sk_lr": SkLogisticRegression(max_iter=1000),
    "sk_svm": SVC(kernel="rbf", C=3, gamma=0.1),
    "sk_dtl": DecisionTreeClassifier(max_depth=8, random_state=Settings.SEED),
    "sk_nb": GaussianNB(),
}

## SVM Feature Selection

SVM diselesaikan dalam bentuk dual dengan SMO, sehingga untuk kernel linear w = Σ αᵢyᵢxᵢ. Fitur diranking berdasarkan |w| SVM linear yang dilatih pada seluruh fitur, lalu `select` fitur teratas dipakai untuk melatih ulang SVM. Model final memakai kernel RBF, yang tidak memiliki w eksplisit.

In [ ]:
prep_display = preprocessor_factory()
X_train_display = prep_display.fit_transform(X_train, y_train)
svm_fitted = fit(Pipeline(SVM(kernel="linear", C=0.1), select=10), X_train_display, y_train)
pd.Series(X_train_display.columns[svm_fitted.keep], name="svm selected features")

---

# Evaluation <a name="5"></a>

---

## Model Comparison

Holdout memakai seed terpisah dari eksperimen sebelumnya; K-Fold pada `X_train` dipakai untuk memilih model dan hyperparameter. Karena analisis eksplorasi sebelumnya sempat melihat target seluruh data train, skor holdout tetap diagnostik dan bukan estimasi generalisasi yang sepenuhnya independen.

In [ ]:
all_models = {**models, **sk_models}
compare(
    holdout(all_models, X_train, y_train, X_val, y_val, METHOD, preprocessor_factory),
    kfold(all_models, X_train, y_train, METHOD, preprocessor_factory),
)

## Hyperparameter Tuning

Bobot meal/room dipertahankan dari konfigurasi terpilih; grid menguji beberapa nilai k dan threshold dengan macro F1.

In [ ]:
grids = {
    "knn": {"k": [51, 71, 91, 111], "threshold": [0.42, 0.44, 0.46, 0.48]},
    "lr": {"lr": [0.01, 0.1], "n_iter": [1000]},
    "svm": {"kernel": ["rbf"], "C": [1, 3, 5], "gamma": [0.1]},
    "dtl": {"max_depth": [5, 8, None]},
    "bn": {"n_bins": [3, 5, 10]},
}
grid_tables = {
    name: tune(models[name], grid, X_train, y_train, METHOD, preprocessor_factory)
    for name, grid in grids.items()
}
tuned_models = {name: best(models[name], table) for name, table in grid_tables.items()}
grid_tables["knn"].head()

In [ ]:
ranking = compare(
    holdout(tuned_models, X_train, y_train, X_val, y_val, METHOD, preprocessor_factory),
    kfold(tuned_models, X_train, y_train, METHOD, preprocessor_factory),
)
ranking

## Error Analysis

In [ ]:
# Pakai baseline scikit-learn jika belum ada model from scratch yang bisa dijalankan.
if ranking.empty:
    ranking = compare(
        holdout(sk_models, X_train, y_train, X_val, y_val, METHOD, preprocessor_factory),
        kfold(sk_models, X_train, y_train, METHOD, preprocessor_factory),
    )
    tuned_models = {**tuned_models, **sk_models}
best_name = ranking.index[0]
best_model = tuned_models[best_name]
best_name, best_model

In [ ]:
confusion(tuned_models, X_train, y_train, X_val, y_val, METHOD, preprocessor_factory)

In [ ]:
report(tuned_models, X_train, y_train, X_val, y_val, METHOD, preprocessor_factory)

In [ ]:
wrong_rows = misclassified(
    best_model, full_train, X_train, y_train, X_val, y_val, METHOD, preprocessor_factory
)
print(f"{len(wrong_rows)} dari {len(X_val)} baris salah")
wrong_rows["predicted"].map({1: "false positive", 0: "false negative"}).value_counts()

In [ ]:
importance(sk_models["sk_dtl"], X_train_display, y_train).head(10)

#### Insights

> TODO

## Bonus Bayesian Network

Tiga bonus pada Bayesian Network: **TAN** (optimasi struktur), **graf**, dan **cost-sensitive classification**. Kodenya ada di `src/models/bayes_network.py` (parameter `tree`, `cost_fp`, `cost_fn`) dan `src/models/bayes_graph.py`. Model utama di bagian Modeling tetap Naive Bayes (`BayesNetwork()`).

### TAN (Tree-Augmented Naive Bayes)

Naive Bayes mengasumsikan fitur saling independen jika kelas diketahui, sehingga fitur yang berkaitan (misalnya `weekday_nights` dan `weekend_nights`) dihitung dua kali dan probabilitasnya terlalu yakin. TAN mengoptimasi *struktur* model: setiap fitur boleh punya satu fitur lain sebagai parent. Pasangan dipilih dengan maximum weight spanning tree atas conditional mutual information I(X<sub>i</sub>; X<sub>j</sub> | C), yaitu seberapa bergantung dua fitur setelah kelas diketahui. CPT tiap fitur menjadi P(X<sub>i</sub> | C, parent).

> Referensi: N. Friedman, D. Geiger, dan M. Goldszmidt, "Bayesian Network Classifiers," *Machine Learning*, vol. 29, no. 2-3, hlm. 131-163, 1997. [https://doi.org/10.1023/A:1007465528199](https://doi.org/10.1023/A:1007465528199)

In [ ]:
bn_variants = {
    "bn_nb": BayesNetwork(n_bins=5),
    "bn_tan": BayesNetwork(n_bins=5, tree=True),
}
compare(holdout(bn_variants, X_train, y_train, X_val, y_val, METHOD), kfold(bn_variants, X_train, y_train, METHOD))

### Graf Bayesian Network

Garis putus-putus abu-abu adalah kelas → fitur (ada untuk semua fitur). Panah biru adalah fitur → fitur, hanya ada pada TAN: panah A → B berarti CPT B dihitung per kombinasi kelas dan nilai A, yaitu P(B | kelas, A). Fitur tanpa panah biru masuk adalah root pohon. Arah panah ditentukan oleh root yang dipilih, jadi bukan sebab-akibat.

In [ ]:
from models.bayes_graph import plot_bayes_graph

bn_tan = fit(BayesNetwork(n_bins=5, tree=True), X_train, y_train, METHOD)
fig = plot_bayes_graph(bn_tan, list(X_train.columns), class_name=Settings.TARGET)
fig.savefig(Settings.OUTPUT_DIR / "bayes_graph.png", dpi=160, bbox_inches="tight")
fig

### Cost-sensitive classification

False positive (memprediksi batal, tamu sebenarnya datang) membuat kamar kosong dijual ulang sehingga hotel **overbooked**, dan diasumsikan 2 kali lebih mahal daripada false negative. Penanganan yang berubah hanya **aturan keputusan**, bukan data dan bukan pelatihan. Biaya yang diharapkan memprediksi batal adalah C<sub>FP</sub>(1 - p) dan memprediksi tidak batal adalah C<sub>FN</sub>p, dengan p = P(batal | x). Prediksi batal hanya jika p > C<sub>FP</sub> / (C<sub>FP</sub> + C<sub>FN</sub>) = 2/3, bukan 0,5. Ini diterapkan di `BayesNetwork.predict` lewat `cost_fp` dan `cost_fn`.

> Referensi: C. Elkan, "The Foundations of Cost-Sensitive Learning," *Proc. IJCAI*, 2001, hlm. 973-977. [https://ijcai.org/proceedings/2001/133](https://ijcai.org/proceedings/2001/133)

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score
from validation import fit_predict

cost_variants = {
    "nb": BayesNetwork(n_bins=5),
    "nb + biaya": BayesNetwork(n_bins=5, cost_fp=2),
    "tan": BayesNetwork(n_bins=5, tree=True),
    "tan + biaya": BayesNetwork(n_bins=5, tree=True, cost_fp=2),
}

rows = {}
for name, model in cost_variants.items():
    predicted = fit_predict(model, X_train, y_train, X_val, METHOD)
    tn, fp, fn, tp = confusion_matrix(y_val, predicted).ravel()
    rows[name] = {"FP": fp, "FN": fn, "biaya (2 FP + FN)": 2 * fp + fn, "macro F1": f1_score(y_val, predicted, average="macro")}
pd.DataFrame(rows).T.astype({"FP": int, "FN": int, "biaya (2 FP + FN)": int})

#### Insights Bonus Bayesian Network

- **TAN lebih baik dari Naive Bayes.** Macro F1 K-Fold 0,764 (std 0,004) berbanding 0,709 (std 0,008), selisih yang jauh lebih besar dari simpangan bakunya. Naive Bayes menghitung fitur yang saling berkaitan (misalnya `total_nights`, `weekday_nights`, `weekend_nights`, atau `month_sin` dan `month_cos`) berulang kali, sedangkan TAN memberi tiap fitur satu parent fitur sehingga bukti ganda itu berkurang.
- **Cost-sensitive menurunkan biaya total (2 FP + FN):** dari 1690 ke 1382 pada TAN dan dari 2426 ke 2065 pada Naive Bayes. False positive turun tajam dan false negative naik, sesuai harga yang disepakati (FP dua kali lebih mahal).
- **Macro F1 tidak otomatis ikut naik.** Pada TAN turun (0,763 menjadi 0,752) karena F1 menganggap semua kesalahan sama berat, padahal yang dioptimalkan adalah biaya. Pada Naive Bayes naik sedikit (0,711 menjadi 0,725) karena Naive Bayes semula terlalu sering memprediksi batal (954 FP berbanding 518 FN).
- Angka di atas berasal dari konfigurasi `Preprocessor` pada notebook ini dengan `seed` tetap. Jika konfigurasinya berubah, jalankan ulang sel di atas dan perbarui angkanya.

## Bonus Bayesian Network

Bonus pada Bayesian Network: **TAN** dan **KDB** (optimasi struktur), **graf**, dan **cost-sensitive classification**. Kodenya ada di `src/models/bayes_network.py` (parameter `tree`, `cost_fp`, `cost_fn`) dan `src/models/bayes_graph.py`. Model utama di bagian Modeling tetap Naive Bayes (`BayesNetwork()`).

### TAN (Tree-Augmented Naive Bayes)

Naive Bayes mengasumsikan fitur saling independen jika kelas diketahui, sehingga fitur yang berkaitan (misalnya `weekday_nights` dan `weekend_nights`) dihitung dua kali dan probabilitasnya terlalu yakin. TAN mengoptimasi *struktur* model: setiap fitur boleh punya satu fitur lain sebagai parent. Pasangan dipilih dengan maximum weight spanning tree atas conditional mutual information I(X<sub>i</sub>; X<sub>j</sub> | C), yaitu seberapa bergantung dua fitur setelah kelas diketahui. CPT tiap fitur menjadi P(X<sub>i</sub> | C, parent).

### KDB (k-Dependence Bayes)

TAN hanya memberi satu parent fitur. KDB mengizinkan hingga `k` parent: fitur diurutkan dari I(X; C) terbesar, lalu tiap fitur mengambil `k` fitur sebelumnya dengan I(X<sub>i</sub>; X<sub>j</sub> | C) terbesar sebagai parent. Dipakai `BayesNetwork(k=10)`; pemilihan `k` ada di laporan. Untuk `k` besar, CPT yang terlalu besar disimpan sparse (hanya sel yang terisi data).

> Referensi: M. Sahami, "Learning Limited Dependence Bayesian Classifiers," *Proc. KDD-96*, 1996, hlm. 335-338. [https://www.osti.gov/biblio/421307](https://www.osti.gov/biblio/421307)

> Referensi: N. Friedman, D. Geiger, dan M. Goldszmidt, "Bayesian Network Classifiers," *Machine Learning*, vol. 29, no. 2-3, hlm. 131-163, 1997. [https://doi.org/10.1023/A:1007465528199](https://doi.org/10.1023/A:1007465528199)

In [ ]:
bn_variants = {
    "bn_nb": BayesNetwork(n_bins=5),
    "bn_tan": BayesNetwork(n_bins=5, tree=True),
    "bn_kdb": BayesNetwork(n_bins=5, k=10),
}
compare(holdout(bn_variants, X_train, y_train, X_val, y_val, METHOD), kfold(bn_variants, X_train, y_train, METHOD))

### Graf Bayesian Network

Garis putus-putus biru keabuan adalah kelas → fitur (ada untuk semua fitur). Panah merah adalah fitur → fitur, hanya ada pada TAN: panah A → B berarti CPT B dihitung per kombinasi kelas dan nilai A, yaitu P(B | kelas, A). Fitur tanpa panah merah masuk adalah root pohon (node hijau). Arah panah ditentukan oleh root yang dipilih, jadi bukan sebab-akibat.

In [ ]:
from models.bayes_graph import plot_bayes_graph

bn_tan = fit(BayesNetwork(n_bins=5, tree=True), X_train, y_train, METHOD)
fig = plot_bayes_graph(bn_tan, list(X_train.columns), class_name=Settings.TARGET)
fig.savefig(Settings.OUTPUT_DIR / "bayes_graph.png", dpi=160, bbox_inches="tight")
fig

### Cost-sensitive classification

False positive (memprediksi batal, tamu sebenarnya datang) membuat kamar kosong dijual ulang sehingga hotel **overbooked**, dan diasumsikan 2 kali lebih mahal daripada false negative. Penanganan yang berubah hanya **aturan keputusan**, bukan data dan bukan pelatihan. Biaya yang diharapkan memprediksi batal adalah C<sub>FP</sub>(1 - p) dan memprediksi tidak batal adalah C<sub>FN</sub>p, dengan p = P(batal | x). Prediksi batal hanya jika p > C<sub>FP</sub> / (C<sub>FP</sub> + C<sub>FN</sub>) = 2/3, bukan 0,5. Ini diterapkan di `BayesNetwork.predict` lewat `cost_fp` dan `cost_fn`.

> Referensi: C. Elkan, "The Foundations of Cost-Sensitive Learning," *Proc. IJCAI*, 2001, hlm. 973-977. [https://ijcai.org/proceedings/2001/133](https://ijcai.org/proceedings/2001/133)

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score
from validation import fit_predict

cost_variants = {
    "nb": BayesNetwork(n_bins=5),
    "nb + biaya": BayesNetwork(n_bins=5, cost_fp=2),
    "tan": BayesNetwork(n_bins=5, tree=True),
    "tan + biaya": BayesNetwork(n_bins=5, tree=True, cost_fp=2),
    "kdb": BayesNetwork(n_bins=5, k=10),
    "kdb + biaya": BayesNetwork(n_bins=5, k=10, cost_fp=2),
}

rows = {}
for name, model in cost_variants.items():
    predicted = fit_predict(model, X_train, y_train, X_val, METHOD)
    tn, fp, fn, tp = confusion_matrix(y_val, predicted).ravel()
    rows[name] = {"FP": fp, "FN": fn, "biaya (2 FP + FN)": 2 * fp + fn, "macro F1": f1_score(y_val, predicted, average="macro")}
pd.DataFrame(rows).T.astype({"FP": int, "FN": int, "biaya (2 FP + FN)": int})

#### Insights Bonus Bayesian Network

- **TAN lebih baik dari Naive Bayes.** Macro F1 K-Fold 0,764 (std 0,004) berbanding 0,709 (std 0,008), selisih yang jauh lebih besar dari simpangan bakunya. Naive Bayes menghitung fitur yang saling berkaitan (misalnya `total_nights`, `weekday_nights`, `weekend_nights`, atau `month_sin` dan `month_cos`) berulang kali, sedangkan TAN memberi tiap fitur satu parent fitur sehingga bukti ganda itu berkurang.
- **Cost-sensitive menurunkan biaya total (2 FP + FN):** dari 1690 ke 1382 pada TAN dan dari 2426 ke 2065 pada Naive Bayes. False positive turun tajam dan false negative naik, sesuai harga yang disepakati (FP dua kali lebih mahal).
- **Macro F1 tidak otomatis ikut naik.** Pada TAN turun (0,763 menjadi 0,752) karena F1 menganggap semua kesalahan sama berat, padahal yang dioptimalkan adalah biaya. Pada Naive Bayes naik sedikit (0,711 menjadi 0,725) karena Naive Bayes semula terlalu sering memprediksi batal (954 FP berbanding 518 FN).
- Angka di atas berasal dari konfigurasi `Preprocessor` pada notebook ini dengan `seed` tetap. Jika konfigurasinya berubah, jalankan ulang sel di atas dan perbarui angkanya.

---

# Deployment <a name="6"></a>

---

Konfigurasi KNN dipilih dengan K-Fold pada data train. Preprocessor lalu di-fit ulang pada seluruh train, dipakai untuk mentransformasi train dan test, lalu disimpan bersama model.


In [ ]:
X_full = full_train.drop(columns=[Settings.ID, Settings.TARGET])
y_full = full_train[Settings.TARGET]
prep_final = preprocessor_factory()
X_full_prepared = prep_final.fit_transform(X_full, y_full)
X_test_prepared = prep_final.transform(test.drop(columns=[Settings.ID]))
submission_name = "knn"
submission_model = tuned_models[submission_name]
submit(
    submission_model,
    submission_name,
    X_full_prepared,
    y_full,
    X_test_prepared,
    test[Settings.ID],
    METHOD,
    preprocessor=prep_final,
)